# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [97]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [98]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # assume x is in shape (B, C, H, W)
    # check if images can be patchified
    B, C, H, W = x.shape
    assert H % patch_size == 0 and W % patch_size == 0

    nh, nw = H // patch_size, W // patch_size
    x = x.reshape(B, C, nh, patch_size, nw, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)
    return x.reshape(B, nh * nw, -1)

x = torch.arange(16).reshape(4, 4).unsqueeze(0).unsqueeze(0)
patchify(x, 2)

tensor([[[ 0,  1,  4,  5],
         [ 2,  3,  6,  7],
         [ 8,  9, 12, 13],
         [10, 11, 14, 15]]])

In [99]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        self.embed_layer = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.embed_layer(x_patches)


class PositionalEmbedding(nn.Module):
    # 1-dimensional positional embedding

    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.n = num_tokens
        assert d_model % 2 == 0  # check if the model dimension is even
        self.d = d_model
        
        p = torch.arange(self.n).unsqueeze(-1)  # (n, 1)
        i = torch.arange(self.d // 2).unsqueeze(0)  # (1, d / 2)
        # p * i broadcasts to (n, d / 2)
        sin = torch.sin(p / 10000 ** (2 * i / self.d))
        cos = torch.cos(p / 10000 ** (2 * i / self.d))
        # stack, permute, reshape to interleave sin and cos
        # will flatten from the last dimension (row-major)
        emb = torch.stack([sin, cos]).permute(1, 2, 0).reshape(self.n, -1)
        self.register_buffer("emb", emb)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.emb
    
x = torch.arange(16).reshape(4, 4)
pe = PositionalEmbedding(4, 4)
pe.emb, pe(x)

(tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998],
         [ 0.1411, -0.9900,  0.0300,  0.9996]]),
 tensor([[ 0.0000,  2.0000,  2.0000,  4.0000],
         [ 4.8415,  5.5403,  6.0100,  7.9999],
         [ 8.9093,  8.5839, 10.0200, 11.9998],
         [12.1411, 12.0100, 14.0300, 15.9995]]))

In [100]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        self.up_gated = nn.Linear(d_model, d_ff_gated, bias=False)
        self.up = nn.Linear(d_model, d_ff_gated, bias=False)
        self.dropout1 = nn.Dropout(dropout)
        self.down = nn.Linear(d_ff_gated, d_model, bias=False)
        self.dropout2 = nn.Dropout(dropout)
        if variant == "GLU":
            self.activation = nn.Sigmoid()
        elif variant == "Bilinear":
            self.activation = nn.Identity()
        elif variant == "ReGLU":
            self.activation = nn.ReLU()
        elif variant == "GEGLU":
            self.activation = nn.GELU()
        else:
            raise KeyError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dropout1(self.activation(self.up_gated(x)) * self.up(x))
        return self.dropout2(self.down(x))

In [101]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = mlp
        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = self.ln1(x)
        x = x + self.dropout(self.attn(norm_x, norm_x, norm_x)[0])  # self-attention
        return x + self.dropout(self.mlp(self.ln2(x)))
    

x = torch.arange(12, dtype=torch.float32).reshape(3, 4).unsqueeze(0)
attn = nn.MultiheadAttention(4, 2, batch_first=True)
out, weight = attn(x, x, x, average_attn_weights=False)
out, weight

(tensor([[[ -6.4849, -10.1207,  -7.3139,  -1.2469],
          [ -5.0323,  -7.7533,  -4.6088,  -0.8535],
          [ -4.4161,  -6.7491,  -3.4614,  -0.6866]]],
        grad_fn=<TransposeBackward0>),
 tensor([[[[1.1746e-08, 1.0837e-04, 9.9989e-01],
           [1.7176e-21, 4.1444e-11, 1.0000e+00],
           [2.5115e-34, 1.5848e-17, 1.0000e+00]],
 
          [[2.0645e-01, 3.1449e-01, 4.7907e-01],
           [6.4367e-01, 2.5517e-01, 1.0116e-01],
           [8.9782e-01, 9.2624e-02, 9.5555e-03]]]], grad_fn=<ViewBackward0>))

In [102]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        # TODO: implement a strategy for embedding the patches
        self.pos = PositionalEmbedding(self.num_tokens + 1, d_model)
        self.embed = PatchEmbed(patch_dim, d_model)
        self.cls = nn.Parameter(torch.randn((1, 1, patch_dim)))  # learnable class embedding

        # TODO: implement a strategy to select the right mlp version for your experiment
        def make_mlp():
            if mlp_kind == "FFN":
                mlp = FeedForward(d_model, d_ff, dropout)
            elif mlp_kind in ["GLU", "Bilinear", "ReGLU", "GEGLU"]:
                mlp = GLUFeedForward(d_model, d_ff // 3 * 2, dropout, mlp_kind)
            else:
                raise KeyError
            return mlp

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=make_mlp(), # TODO: Feed your mlp to the encoder blocks
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        self.out_proj = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = patchify(x, self.patch_size)
        # add a learnable class token to the last of every sequence
        x = torch.concat([x, self.cls.repeat(x.size(0), 1, 1)], dim=1)
        x = self.embed(x)
        x = self.pos(x)
        for block in self.blocks:
            x = block(x)
        logits = self.out_proj(x[:, -1])  # readout from the cls token
        return logits

In [103]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cuda"  # set "cuda" if available

In [104]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = nn.functional.cross_entropy(logits, yb) # TODO: Your criterion

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    return {
        # TODO: Return your metrics that you think will support your claim for this experiment
    }

In [105]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="mps")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = ["FFN", "GLU"] # TODO: Name your runs
results = []

for kind in runs:
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    # TODO: print anything you might want here
    print(f"\nRun: {kind} | " )
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)


Run: FFN | 
[FFN] epoch 1/5 | test acc: 0.8715
[FFN] epoch 2/5 | test acc: 0.9203
[FFN] epoch 3/5 | test acc: 0.9400
[FFN] epoch 4/5 | test acc: 0.9501
[FFN] epoch 5/5 | test acc: 0.9499

Run: GLU | 
[GLU] epoch 1/5 | test acc: 0.8762
[GLU] epoch 2/5 | test acc: 0.9151
[GLU] epoch 3/5 | test acc: 0.9332
[GLU] epoch 4/5 | test acc: 0.9426
[GLU] epoch 5/5 | test acc: 0.9517
